# 1.挂载Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# 验证模型文件夹存在
import os
MODEL_PATH = '/content/drive/MyDrive/Code/BIT-Control-Project/models/cogvlm2-llama3-chat-19B'  # 根据实际路径修改

Mounted at /content/drive


# 2.安装依赖

In [2]:
# 卸载所有相关包
!pip uninstall torch torchvision torchaudio xformers -y -q 2>/dev/null

# 安装 PyTorch 2.4 + CUDA 12.1（最稳定组合）
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121 -q

# 安装匹配的 xformers
!pip install xformers==0.0.27.post2 --index-url https://download.pytorch.org/whl/cu121 -q

# 其他依赖
!pip install flask pyngrok bitsandbytes sentencepiece transformers==4.41.0 accelerate==0.30.0 -q

# 验证
import torch, xformers
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ xformers: {xformers.__version__}")
print(f"✅ CUDA: {torch.version.cuda}")
print("✅ 依赖安装完成，请继续运行 Cell 3")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 600.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 128.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 96.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 108.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 67.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 133.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 595.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 15.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━

# 3.加载CogVLM2模型

In [3]:
import os
os.environ["XFORMERS_DISABLED"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = '/content/drive/MyDrive/Code/BIT-Control-Project/models/cogvlm2-llama3-chat-19B'
TORCH_TYPE = torch.bfloat16

print(f'🔄 正在加载 CogVLM2...')
print(f'PyTorch: {torch.__version__}')

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=TORCH_TYPE,
    trust_remote_code=True,
    load_in_4bit=True,
    device_map='auto',
    local_files_only=True
)

model = model.eval()
print(f'✅ CogVLM2 加载完成！显存: {torch.cuda.memory_allocated()/1024**3:.2f}GB')

# import os
# os.environ["XFORMERS_DISABLED"] = "1"
# os.environ["TOKENIZERS_PARALLELISM"] = "false"

# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer

# MODEL_PATH = '/content/drive/MyDrive/Code/BIT-Control-Project/models/cogvlm2-llama3-chat-19B'
# TORCH_TYPE = torch.bfloat16

# print(f'🔄 正在加载 CogVLM2...')

# tokenizer = AutoTokenizer.from_pretrained(
#     MODEL_PATH,
#     trust_remote_code=True,
#     local_files_only=True
# )

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_PATH,
#     torch_dtype=TORCH_TYPE,
#     trust_remote_code=True,
#     device_map='auto',
#     local_files_only=True,
#     low_cpu_mem_usage=True
# )

# model = model.eval()
# print(f'✅ CogVLM2 加载完成！显存: {torch.cuda.memory_allocated()/1024**3:.2f}GB')

🔄 正在加载 CogVLM2...
PyTorch: 2.4.0+cu121


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

✅ CogVLM2 加载完成！显存: 11.79GB


# 4.启动API服务器

In [ ]:
ngrok.kill()

In [ ]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading
import time
import base64
import io
from PIL import Image
import torch

# 定义设备和数据类型（确保与 cell-5 一致）
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TORCH_TYPE = torch.bfloat16

# ✅ 配置你的 ngrok authtoken
NGROK_TOKEN = '397eqzP33VdhCluHPwgX3wFaKzv_4z4KcGAFzC5Hyhtr4LD1f'  # 替换为你的 token
ngrok.set_auth_token(NGROK_TOKEN)

app = Flask(__name__)

def decode_base64_image(base64_str):
    """从 base64 字符串解码图像"""
    if ',' in base64_str:
        base64_str = base64_str.split(',')[1]
    img_data = base64.b64decode(base64_str)
    return Image.open(io.BytesIO(img_data)).convert('RGB')

def generate_response(prompt, image=None):
    """生成模型响应"""
    try:
        if image is not None:
            raw_inputs = model.build_conversation_input_ids(
                tokenizer,
                query=prompt,
                images=[image],
                template_version='chat'
            )
        else:
            raw_inputs = model.build_conversation_input_ids(
                tokenizer,
                query=prompt,
                images=None,
                template_version='chat'
            )

        # 构造模型输入（修复：先保存 images 再构造新字典）
        inputs = {
            'input_ids': raw_inputs['input_ids'].unsqueeze(0).to(DEVICE),
            'token_type_ids': raw_inputs['token_type_ids'].unsqueeze(0).to(DEVICE),
            'attention_mask': raw_inputs['attention_mask'].unsqueeze(0).to(DEVICE),
        }
        
        # 从原始输入中获取 images
        if 'images' in raw_inputs and raw_inputs['images'] is not None:
            inputs['images'] = [[raw_inputs['images'][0].to(DEVICE).to(TORCH_TYPE)]]

        gen_kwargs = {
            'max_new_tokens': 512,
            'do_sample': True,
            'temperature': 0.7,
            'top_p': 0.9,
        }

        with torch.no_grad():
            outputs = model.generate(**inputs, **gen_kwargs)
            outputs = outputs[:, inputs['input_ids'].shape[1]:]
            response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        return response.strip()

    except Exception as e:
        print(f'❌ Generation error: {e}')
        import traceback
        traceback.print_exc()
        return f'Error: {str(e)}'

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'healthy', 'model': 'cogvlm2'})

@app.route('/v1/chat/completions', methods=['POST'])
def chat_completions():
    try:
        data = request.json
        messages = data.get('messages', [])
        prompt = ''
        image = None

        for msg in messages:
            if msg['role'] == 'user':
                content = msg['content']
                if isinstance(content, str):
                    prompt = content
                elif isinstance(content, list):
                    for item in content:
                        if item['type'] == 'text':
                            prompt = item['text']
                        elif item['type'] == 'image_url':
                            img_url = item['image_url']['url']
                            if img_url.startswith('data:'):
                                image = decode_base64_image(img_url)

        start_time = time.time()
        response = generate_response(prompt, image)
        elapsed = time.time() - start_time
        print(f'[CogVLM2] 推理耗时: {elapsed:.2f}s')

        return jsonify({
            'id': 'chatcmpl-cogvlm2',
            'object': 'chat.completion',
            'created': int(time.time()),
            'model': 'cogvlm2',
            'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': response}, 'finish_reason': 'stop'}],
            'usage': {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0}
        })

    except Exception as e:
        print(f'❌ API Error: {e}')
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

public_url = ngrok.connect(5000)
print('=' * 60)
print(f'📡 CogVLM2 API 服务地址: {public_url}')
print('=' * 60)

threading.Thread(target=lambda: app.run(port=5000, use_reloader=False)).start()